In [5]:
import pandas as pd
import numpy as np
import sqlite3
from scipy import stats

print("Analysis libraries imported successfully!")

Analysis libraries imported successfully!


In [6]:

import pandas as pd

transactions = pd.read_csv("https://raw.githubusercontent.com/naisoisharonkarinoh/plp-capstone-retail-sharon-koina/main/data/transactions.csv")
products = pd.read_csv("https://raw.githubusercontent.com/naisoisharonkarinoh/plp-capstone-retail-sharon-koina/main/data/products.csv")
branches = pd.read_csv("https://raw.githubusercontent.com/naisoisharonkarinoh/plp-capstone-retail-sharon-koina/main/data/branches.csv")
customers = pd.read_csv("https://raw.githubusercontent.com/naisoisharonkarinoh/plp-capstone-retail-sharon-koina/main/data/customers.csv")

print("All four CSV files loaded successfully!")
print("Transactions:", transactions.shape)
print("Products:", products.shape)
print("Branches:", branches.shape)
print("Customers:", customers.shape)


All four CSV files loaded successfully!
Transactions: (50000, 8)
Products: (20, 4)
Branches: (5, 2)
Customers: (5000, 2)


In [7]:
# SQL analysis: Revenue by supermarket branch

query = """
SELECT
    b.branch_name,
    SUM(t.sales_amount) AS total_revenue,
    COUNT(DISTINCT t.transaction_id) AS total_transactions,
    SUM(t.quantity) AS units_sold
FROM transactions AS t
JOIN branches AS b
    ON t.branch_id = b.branch_id
GROUP BY b.branch_name
ORDER BY total_revenue DESC;
"""

branch_sql_results = pd.read_sql(query, connection)

display(branch_sql_results)

,branch_name,total_revenue,total_transactions,units_sold
0,Kisumu,7756715,10102,55687
1,Nakuru,7687115,9964,54811
2,Mombasa,7669890,10025,54814
3,Nairobi,7637720,9947,54317
4,Eldoret,7541125,9962,54514


In [8]:
# SQL analysis: Revenue by product category

query = """
SELECT
    p.category,
    SUM(t.sales_amount) AS total_revenue,
    SUM(t.quantity) AS units_sold,
    COUNT(DISTINCT t.transaction_id) AS total_transactions
FROM transactions AS t
JOIN products AS p
    ON t.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

category_sql_results = pd.read_sql(query, connection)

display(category_sql_results)

,category,total_revenue,units_sold,total_transactions
0,Groceries,17705830,109130,19975
1,Beverages,7839340,55013,9961
2,Household,4828200,27642,5047
3,Breakfast,2964720,13476,2484
4,Personal Care,2080800,13872,2514
5,Dairy,1311605,27797,5023
6,Bakery,872820,13428,2450
7,Snacks,689250,13785,2546


In [9]:
# SQL analysis: Customers generating more than KES 20,000

query = """
SELECT
    t.customer_id,
    c.customer_segment,
    SUM(t.sales_amount) AS total_spending,
    COUNT(DISTINCT t.transaction_id) AS total_transactions
FROM transactions AS t
JOIN customers AS c
    ON t.customer_id = c.customer_id
GROUP BY
    t.customer_id,
    c.customer_segment
HAVING SUM(t.sales_amount) > 20000
ORDER BY total_spending DESC;
"""

high_value_customers = pd.read_sql(query, connection)

print("Customers with total spending above KES 20,000:")

display(high_value_customers.head(10))

print(
    "\nTotal customers meeting this condition:",
    len(high_value_customers)
)

Customers with total spending above KES 20,000:


,customer_id,customer_segment,total_spending,total_transactions
0,3664,Regular,21530,16
1,2675,Premium,21185,14
2,2018,Premium,20930,21
3,2949,Regular,20800,15
4,96,Regular,20670,17



Total customers meeting this condition: 5


In [10]:
# SQL analysis: Rank branches by revenue

query = """
WITH branch_revenue AS (
    SELECT
        b.branch_name,
        SUM(t.sales_amount) AS total_revenue
    FROM transactions AS t
    JOIN branches AS b
        ON t.branch_id = b.branch_id
    GROUP BY b.branch_name
)
SELECT
    branch_name,
    total_revenue,
    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank
FROM branch_revenue
ORDER BY revenue_rank;
"""

branch_rankings = pd.read_sql(query, connection)

print("BRANCH REVENUE RANKINGS")
display(branch_rankings)

BRANCH REVENUE RANKINGS


,branch_name,total_revenue,revenue_rank
0,Kisumu,7756715,1
1,Nakuru,7687115,2
2,Mombasa,7669890,3
3,Nairobi,7637720,4
4,Eldoret,7541125,5


In [11]:
# Supermarket Business KPI Scorecard

total_revenue = transactions["sales_amount"].sum()

total_transactions = transactions["transaction_id"].nunique()

total_units_sold = transactions["quantity"].sum()

average_transaction_value = (
    total_revenue / total_transactions
)

unique_customers = transactions["customer_id"].nunique()

average_revenue_per_customer = (
    total_revenue / unique_customers
)

kpi_scorecard = pd.DataFrame({
    "KPI": [
        "Total Revenue",
        "Total Transactions",
        "Total Units Sold",
        "Average Transaction Value",
        "Unique Customers",
        "Average Revenue per Customer"
    ],
    "Value": [
        round(total_revenue, 2),
        total_transactions,
        total_units_sold,
        round(average_transaction_value, 2),
        unique_customers,
        round(average_revenue_per_customer, 2)
    ],
    "Definition": [
        "Sum of all transaction sales amounts",
        "Number of unique transactions",
        "Sum of quantities sold",
        "Total revenue divided by transactions",
        "Number of distinct purchasing customers",
        "Total revenue divided by unique customers"
    ]
})

display(kpi_scorecard)

,KPI,Value,Definition
0,Total Revenue,38292565.00,Sum of all transaction sales amounts
1,Total Transactions,50000.00,Number of unique transactions
2,Total Units Sold,274143.00,Sum of quantities sold
3,Average Transaction Value,765.85,Total revenue divided by transactions
4,Unique Customers,4999.00,Number of distinct purchasing customers
5,Average Revenue per Customer,7660.05,Total revenue divided by unique customers


In [12]:
# Statistical test: Nairobi vs Mombasa

# Select transaction values for each branch
nairobi_sales = transactions.loc[
    transactions["branch_id"] == 1, "sales_amount"
]

mombasa_sales = transactions.loc[
    transactions["branch_id"] == 4, "sales_amount"
]

# Calculate average transaction values
nairobi_average = nairobi_sales.mean()
mombasa_average = mombasa_sales.mean()

# Perform Welch's independent two-sample t-test
t_statistic, p_value = stats.ttest_ind(
    nairobi_sales,
    mombasa_sales,
    equal_var=False
)

# Display results
print("STATISTICAL TEST: NAIROBI VS MOMBASA")
print("------------------------------------")
print(f"Nairobi average transaction value: KES {nairobi_average:,.2f}")
print(f"Mombasa average transaction value: KES {mombasa_average:,.2f}")
print(f"Difference: KES {nairobi_average - mombasa_average:,.2f}")
print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

# Interpret the result
if p_value < 0.05:
    print("\nConclusion: The difference is statistically significant.")
else:
    print("\nConclusion: There is insufficient evidence of a difference.")

STATISTICAL TEST: NAIROBI VS MOMBASA
------------------------------------
Nairobi average transaction value: KES 767.84
Mombasa average transaction value: KES 765.08
Difference: KES 2.77
T-statistic: 0.2944
P-value: 0.7685

Conclusion: There is insufficient evidence of a difference.


In [13]:
import sqlite3

connection = sqlite3.connect(":memory:")

transactions.to_sql("transactions", connection, index=False, if_exists="replace")
branches.to_sql("branches", connection, index=False, if_exists="replace")
products.to_sql("products", connection, index=False, if_exists="replace")
customers.to_sql("customers", connection, index=False, if_exists="replace")

print("Database connection created successfully!")

Database connection created successfully!
